## Built-in validation (types, ranges, patterns)

FastAPI automatically validates request data using Python type hints and Pydantic models.

### Type validation

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    quantity: int

@app.post("/items/")
def create_item(item: Item):
    return {"item": item}

- If a client sends a string for price instead of a float, FastAPI rejects the request automatically.

- No manual type checking is needed

### Value constraints

Pydantic supports range constraints and patterns:

In [ ]:
from pydantic import BaseModel, Field, constr

class Item(BaseModel):
    name: constr(min_length=3, max_length=50)  # string length
    price: float = Field(gt=0)                  # greater than 0
    quantity: int = Field(ge=1, le=100)        # between 1 and 100

- `gt`, `ge`, `lt`, `le` → greater than, greater/equal, less than, less/equal

- constr(pattern="^[A-Z]+$") → regex pattern

### Validation errors and how FastAPI returns them

- When validation fails, FastAPI responds with HTTP 422 Unprocessable Entity.

- The response includes details about the field and error.

Example request with invalid data:
`{`
 ` "name": "A",`
 ` "price": -5,`
`  "quantity": 200`
`}`


#### Response

In [ ]:


{
  "detail": [
    {
      "loc": ["body", "name"],
      "msg": "ensure this value has at least 3 characters",
      "type": "value_error.any_str.min_length"
    },
    {
      "loc": ["body", "price"],
      "msg": "ensure this value is greater than 0",
      "type": "value_error.number.not_gt"
    },
    {
      "loc": ["body", "quantity"],
      "msg": "ensure this value is less than or equal to 100",
      "type": "value_error.number.not_le"
    }
  ]
}

- `loc` → location of the error (body, query, path, etc.)

- `msg` → human-readable explanation

- `type` → error type

### HTTPException and custom error responses

FastAPI allows raising HTTPException for custom errors.

In [ ]:
from fastapi import HTTPException

@app.get("/items/{item_id}")
def read_item(item_id: int):
    if item_id > 10:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"item_id": item_id}

- `status_code` → the HTTP status code to return

- `detail` → message that will appear in the response

**Custom response example:**

In [ ]:
from fastapi.responses import JSONResponse

@app.get("/custom_error/")
def custom_error():
    return JSONResponse(status_code=400, content={"message": "Custom bad request"})

| Status Code | Use Case |
|------------|----------|
| 200 OK | Successful GET, PUT, DELETE (data returned) |
| 201 Created | Successful POST (resource created) |
| 400 Bad Request | Client sent invalid request (missing fields, wrong data type) |
| 404 Not Found | Resource not found (e.g., invalid ID) |
| 422 Unprocessable Entity | Validation failed (automatic by FastAPI for Pydantic) |
| 500 Internal Server Error | Server-side failure, unexpected error |

### Common API errors and how to handle them

#### 1. **Validation errors**

Handled automatically by FastAPI (422)

Ensure Pydantic models are correctly defined

#### 2. **Not found errors**

In [ ]:
@app.get("/products/{product_id}")
def get_product(product_id: int):
    product = db.get(product_id)
    if not product:
        raise HTTPException(status_code=404, detail="Product not found")
    return product

 #### 3. **Bad requests**

When input is invalid but technically valid JSON

In [ ]:
@app.post("/divide/")
def divide(a: int, b: int):
    if b == 0:
        raise HTTPException(status_code=400, detail="Division by zero is not allowed")
    return {"result": a / b}

#### 4. **Internal server errors**

- Unhandled exceptions automatically return 500

- Use try/except blocks to provide more informative responses:

In [ ]:
@app.get("/read_file/")
def read_file():
    try:
        with open("data.txt") as f:
            return {"content": f.read()}
    except FileNotFoundError:
        raise HTTPException(status_code=404, detail="File not found")
    except Exception:
        raise HTTPException(status_code=500, detail="Internal server error")

## FastAPI Validation and Error Handling

- **Type validation + Pydantic constraints** → automatic **422 Unprocessable Entity** errors
- **HTTPException** → custom error messages and status codes
- **Use proper HTTP status codes for clarity:**
  - **2xx** → success
  - **4xx** → client errors
  - **5xx** → server errors
- Always **validate input** and **handle exceptions gracefully**
- **Custom responses** can improve client experience